# IV Mean Reversion — Full Pipeline Notebook
Pipeline: **Load IB data → Signals → Backtest → Live scan**

- Real data loaded from `data/iv_raw/*.parquet` (IB download via `ibkr_iv_downloader.py`)
- yfinance fallback for any ticker missing from local store
- Signal engine, backtester, and live scanner match the production scripts exactly

In [3]:
import sys
from pathlib import Path

# Make src/ and scripts/ importable
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
! pip install seaborn
import seaborn as sns
! pip install yfinance
import yfinance as yf

from iv_mr.iv_utils import iv_zscore, iv_percentile, half_life
from signal_engine import compute_signals, scan_universe, ENTRY_DISC, PROFIT_TARGET, MAX_HOLD_DAYS
from backtester import backtest_ticker, build_equity_curve, print_stats, plot_results, CAPITAL, POSITION_SIZE, COMMISSION_PCT, START_DATE

DATA_DIR = ROOT / "data" / "iv_raw"
sns.set_theme(style="darkgrid")
print(f"ROOT : {ROOT}")
print(f"DATA : {DATA_DIR}  ({len(list(DATA_DIR.glob('*.parquet')))} parquet files found)")
print("Imports OK ✅")


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.5-py3-none-any.whl.metadata (8.6 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.5-py3-none-any.whl.metadata (8.6 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached beautifulsoup4-4.14.3-py3-non

ModuleNotFoundError: No module named 'iv_mr'

## 1. Load IV + Price Data
Loads real IB data from `data/iv_raw/<TICKER>.parquet`.  
**Schema**: `[open, high, low, close, volume, iv_annual]` — `iv_annual` is annualised IV as a decimal (e.g. `0.25` = 25%).  
**Fallback**: if the parquet file is missing, downloads 2y of price from yfinance and proxies IV with 21-day realised vol.

In [5]:
SYMBOL = "SPY"   # change to any ticker with a parquet in data/iv_raw/

from pathlib import Path

DATA_DIR = Path("../data/iv_raw")
SYMBOL = "SPY"  # or whichever symbol you're working with

parquet_path = DATA_DIR / f"{SYMBOL}.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"✅ Loaded IB data: {len(df)} rows  ({df.index[0].date()} → {df.index[-1].date()})")
    print(f"   Columns : {df.columns.tolist()}")
    print(f"   IV range: {df['iv_annual'].min():.3f} – {df['iv_annual'].max():.3f}")
else:
    print(f"⚠  No parquet for {SYMBOL} — using yfinance + realised-vol proxy")
    raw = yf.download(SYMBOL, period="2y", auto_adjust=True)
    log_ret = np.log(raw["Close"] / raw["Close"].shift(1))
    rv = log_ret.rolling(21).std() * np.sqrt(252)
    df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.columns = ["open", "high", "low", "close", "volume"]
    df["iv_annual"] = rv
    df = df.dropna()
    print(f"   yfinance rows: {len(df)}")

df.tail(3)

⚠  No parquet for SPY — using yfinance + realised-vol proxy


[*********************100%***********************]  1 of 1 completed

   yfinance rows: 479


,open,high,low,close,volume,iv_annual
Date,,,,,,
2026-04-23,709.500000,712.359985,702.280029,708.450012,56174000,0.178625
2026-04-24,710.750000,714.469971,709.010010,713.940002,45123600,0.179040
2026-04-27,713.169983,715.239990,712.294983,715.150085,21338148,0.160605


## 2. IV Mean-Reversion Signals
Uses production `compute_signals()` from `scripts/signal_engine.py`.  
Key derived columns: `div` (daily IV), `daily_move`, `below_prev_low`, `exceeds_div`, `entry_signal`, `limit_entry`, `profit_target`.

In [9]:
import sys
from pathlib import Path

ROOT = Path("/Users/diegoochoa/IV_mean_reversion")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

print(sys.path[:3])  # verify paths are correct
from signal_engine import compute_signals, scan_universe, ENTRY_DISC, PROFIT_TARGET, MAX_HOLD_DAYS
print("✅ signal_engine imported")

signals_df = compute_signals(df)

iv = signals_df["iv_annual"]
z   = iv_zscore(iv, window=20)
pct = iv_percentile(iv, window=252)
hl  = half_life(iv)

print(f"Half-life of IV mean reversion : {hl:.1f} trading days")
print(f"Entry signals in dataset       : {signals_df['entry_signal'].sum()}")
print(f"Signal rate                    : {signals_df['entry_signal'].mean():.2%}")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

iv_pct = iv * 100  # display as %
iv_pct.plot(ax=axes[0], title=f"{SYMBOL} — IV Annual (%)", color="steelblue")

z.plot(ax=axes[1], title="IV Rolling Z-Score (20d)", color="darkorange")
axes[1].axhline( 1.5, ls="--", color="red",   lw=0.9, label="z=+1.5")
axes[1].axhline(-1.5, ls="--", color="green", lw=0.9, label="z=-1.5")
axes[1].legend(fontsize=8)

# Mark entry signals
entry_dates = signals_df[signals_df["entry_signal"]].index
axes[1].scatter(entry_dates, z.loc[entry_dates], color="red", s=20, zorder=5, label="Entry signal")

pct.plot(ax=axes[2], title="IV Percentile Rank (252d)", color="purple")
axes[2].axhline(80, ls="--", color="red",   lw=0.9)
axes[2].axhline(20, ls="--", color="green", lw=0.9)

plt.tight_layout()
plt.show()

['/Users/diegoochoa/IV_mean_reversion/scripts', '/Users/diegoochoa/IV_mean_reversion/src', '/Users/diegoochoa/scripts']
✅ signal_engine imported


NameError: name 'iv_zscore' is not defined

## 3. Backtest — Single Ticker
Uses production `backtest_ticker()` + `build_equity_curve()` + `print_stats()` from `scripts/backtester.py`.  
Strategy: **limit buy** at `close × (1 − 0.5%)` → **profit target** at `+4%` → **time stop** after 7 days.  
Commission: 0.1% per leg (round-trip 0.2%). Capital: $100k, position size: 5%.

In [ ]:
bt_df = signals_df[signals_df.index >= START_DATE]

trades = backtest_ticker(bt_df, SYMBOL)
trades_df = pd.DataFrame(trades)

if trades_df.empty:
    print("No trades found for this ticker / period.")
else:
    equity = build_equity_curve(trades_df)
    print_stats(trades_df, equity)
    plot_results(equity)
    display(trades_df.tail(10))

## 4. Backtest — Full Universe
Runs `backtest_ticker()` across all parquet files in `data/iv_raw/`, combines results, and plots the aggregate equity curve.

In [ ]:
all_trades = []

for f in sorted(DATA_DIR.glob("*.parquet")):
    ticker = f.stem
    try:
        raw = pd.read_parquet(f)
        raw = raw[raw.index >= START_DATE]
        if len(raw) < 250:
            continue
        t = backtest_ticker(raw, ticker)
        all_trades.extend(t)
    except Exception as e:
        print(f"  ⚠ {ticker}: {e}")

all_trades_df = pd.DataFrame(all_trades)

if all_trades_df.empty:
    print("No trades — run the downloader first: python scripts/test_download.py")
else:
    all_trades_df = all_trades_df.sort_values("entry_date")
    universe_equity = build_equity_curve(all_trades_df)
    print_stats(all_trades_df, universe_equity)
    plot_results(universe_equity)

    # Exit reason breakdown
    print("\nExit reason breakdown:")
    print(all_trades_df["exit_reason"].value_counts().to_string())

    # Per-ticker win rate
    per_ticker = (
        all_trades_df.groupby("ticker")["return_net"]
        .agg(["count", "mean", lambda x: (x > 0).mean()])
        .rename(columns={"count": "trades", "mean": "avg_ret", "<lambda_0>": "win_rate"})
        .sort_values("avg_ret", ascending=False)
    )
    print("\nTop 10 tickers by avg return:")
    display(per_ticker.head(10))

## 5. Live Signal Scan — Today's Signals
Runs `scan_universe()` on all stored parquet files and shows tickers triggering the entry signal today.  
These are the same signals `scripts/live_runner.py` uses to place limit orders.

In [ ]:
live_signals = scan_universe(DATA_DIR)

print(f"{'='*60}")
print(f"  SIGNALS TODAY  ({pd.Timestamp.now().date()})")
print(f"{'='*60}")

if live_signals.empty:
    print("No signals today.")
else:
    print(f"  {len(live_signals)} signal(s) found:\n")
    display(
        live_signals.style
        .format({
            "close":        "${:.2f}",
            "iv_annual":    "{:.1%}",
            "div":          "{:.4f}",
            "daily_move":   "{:.4f}",
            "limit_entry":  "${:.2f}",
            "profit_target":"${:.2f}",
        })
        .background_gradient(subset=["div"], cmap="YlOrRd")
    )